# Colab Qualitative Review

This notebook provides a Colab-friendly manual review flow for generated product images. It reads `results/evaluation_report.csv`, shows one image at a time, and saves manual ratings to `results/quality_scores.csv` plus aggregated results in `results/quality_summary.csv`.

Use this when you want to score outputs inside Colab instead of running the desktop `evaluate_quality.py` tool.

In [ ]:
from pathlib import Path
import csv
import pandas as pd
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets

try:
    from google.colab import drive
except ImportError:
    drive = None

MOUNT_DRIVE = False
REPO_PATH = Path('/content/COMP_SCI_5542/GENAI_Stable_Diffussion_Challenge')
RESULTS_DIR = REPO_PATH / 'results'
OUTPUTS_DIR = REPO_PATH / 'outputs'
REPORT_PATH = RESULTS_DIR / 'evaluation_report.csv'
QUALITY_SCORES_PATH = RESULTS_DIR / 'quality_scores.csv'
QUALITY_SUMMARY_PATH = RESULTS_DIR / 'quality_summary.csv'
MAX_PREVIEW_SIZE = (720, 720)

if MOUNT_DRIVE and drive is not None:
    drive.mount('/content/drive')

if not REPO_PATH.exists():
    raise FileNotFoundError(f'Repo path not found: {REPO_PATH}')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repo path: {REPO_PATH}')
print(f'Results dir: {RESULTS_DIR}')
print(f'Outputs dir: {OUTPUTS_DIR}')
print(f'Review report: {REPORT_PATH}')

In [ ]:
def resolve_image_path(path_str: str) -> Path:
    raw_path = Path(path_str)
    if raw_path.exists():
        return raw_path

    if not raw_path.is_absolute():
        candidate = REPO_PATH / raw_path
        if candidate.exists():
            return candidate

    if raw_path.name:
        matches = list(OUTPUTS_DIR.rglob(raw_path.name))
        if matches:
            return matches[0]

    raise FileNotFoundError(path_str)


def load_review_items():
    items = []
    if REPORT_PATH.exists():
        with open(REPORT_PATH, 'r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                try:
                    image_path = resolve_image_path(row['image_path'])
                except FileNotFoundError:
                    continue
                items.append({
                    'product_id': row.get('product_id', ''),
                    'product_title': row.get('product_title', ''),
                    'prompt_type': row.get('prompt_type', ''),
                    'image_path': str(image_path),
                    'image_name': image_path.name,
                    'prompt': row.get('prompt', ''),
                })
    else:
        for image_path in sorted(OUTPUTS_DIR.rglob('*.png')):
            parts = image_path.stem.split('_')
            items.append({
                'product_id': parts[0] if parts else 'unknown',
                'product_title': parts[0] if parts else 'unknown',
                'prompt_type': parts[1] if len(parts) > 1 else 'unknown',
                'image_path': str(image_path),
                'image_name': image_path.name,
                'prompt': '',
            })
    return items


def load_existing_scores():
    if not QUALITY_SCORES_PATH.exists():
        return {}
    with open(QUALITY_SCORES_PATH, 'r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        return {row['image_name']: row for row in reader if row.get('image_name')}


def save_scores(rows):
    fieldnames = [
        'product_id',
        'product_title',
        'prompt_type',
        'image_name',
        'image_path',
        'quality_score',
        'quality_notes',
    ]
    with open(QUALITY_SCORES_PATH, 'w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_summary(rows):
    if not rows:
        return
    df = pd.DataFrame(rows)
    df['quality_score'] = pd.to_numeric(df['quality_score'], errors='coerce')
    grouped = (
        df.groupby(['product_id', 'product_title', 'prompt_type'], dropna=False)
        .agg(
            mean_quality_score=('quality_score', 'mean'),
            n_quality_scored=('quality_score', lambda values: int(values.notna().sum())),
        )
        .reset_index()
    )
    grouped['mean_quality_score'] = grouped['mean_quality_score'].round(2)
    grouped.to_csv(QUALITY_SUMMARY_PATH, index=False)


review_items = load_review_items()
saved_scores = load_existing_scores()
print(f'Images available for review: {len(review_items)}')
print(f'Existing saved ratings: {len(saved_scores)}')

In [ ]:
if not review_items:
    raise RuntimeError('No review items found. Run generation first or check the results path.')

state = {
    'index': 0,
    'scores': dict(saved_scores),
}

status_html = widgets.HTML()
meta_html = widgets.HTML()
path_html = widgets.HTML()
prompt_html = widgets.HTML()
score_buttons = widgets.ToggleButtons(
    options=[('1', '1'), ('2', '2'), ('3', '3'), ('4', '4'), ('5', '5')],
    description='Score:',
    style={'description_width': 'initial'},
)
notes_box = widgets.Textarea(
    value='',
    description='Notes:',
    placeholder='Optional notes about realism, background, color, or artifacts',
    layout=widgets.Layout(width='100%', height='120px'),
    style={'description_width': 'initial'},
)
image_output = widgets.Output()
message_html = widgets.HTML()

prev_button = widgets.Button(description='Previous', button_style='')
save_button = widgets.Button(description='Save', button_style='success')
next_button = widgets.Button(description='Next', button_style='')
export_button = widgets.Button(description='Export Summary', button_style='info')


def current_item():
    return review_items[state['index']]


def sorted_rows():
    return sorted(
        state['scores'].values(),
        key=lambda row: (row['product_id'], row['prompt_type'], row['image_name'])
    )


def persist_rows():
    rows = sorted_rows()
    save_scores(rows)
    save_summary(rows)


def render_item():
    item = current_item()
    existing = state['scores'].get(item['image_name'], {})

    status_html.value = f'<b>Image {state["index"] + 1} / {len(review_items)}</b>'
    meta_html.value = (
        f'<b>{item["product_id"]}</b> | {item["prompt_type"]} | ' 
        f'{item["product_title"]}'
    )
    path_html.value = f'<code>{item["image_path"]}</code>'
    prompt_html.value = f'<pre style="white-space: pre-wrap;">{item["prompt"]}</pre>' if item['prompt'] else ''

    score_buttons.value = existing.get('quality_score', None) if existing.get('quality_score') else None
    notes_box.value = existing.get('quality_notes', '')

    with image_output:
        clear_output(wait=True)
        image = Image.open(item['image_path']).convert('RGB')
        image.thumbnail(MAX_PREVIEW_SIZE)
        display(image)


def collect_current_row():
    item = current_item()
    return {
        'product_id': item['product_id'],
        'product_title': item['product_title'],
        'prompt_type': item['prompt_type'],
        'image_name': item['image_name'],
        'image_path': item['image_path'],
        'quality_score': score_buttons.value or '',
        'quality_notes': notes_box.value.strip(),
    }


def save_current(_=None):
    row = collect_current_row()
    if not row['quality_score']:
        message_html.value = '<span style="color:#b91c1c;">Choose a score from 1 to 5 before saving.</span>'
        return
    state['scores'][row['image_name']] = row
    persist_rows()
    message_html.value = (
        f'<span style="color:#166534;">Saved {row["image_name"]}. ' 
        f'Total rated: {len(state["scores"])}.</span>'
    )


def go_previous(_=None):
    if state['index'] > 0:
        state['index'] -= 1
        render_item()


def go_next(_=None):
    if state['index'] < len(review_items) - 1:
        state['index'] += 1
        render_item()


def export_summary(_=None):
    persist_rows()
    message_html.value = (
        '<span style="color:#1d4ed8;">Exported quality CSV files. ' 
        'Re-run run_pipeline.py --eval-only to merge them into the evaluation reports.</span>'
    )


prev_button.on_click(go_previous)
save_button.on_click(save_current)
next_button.on_click(go_next)
export_button.on_click(export_summary)

controls = widgets.HBox([prev_button, save_button, next_button, export_button])
panel = widgets.VBox([
    status_html,
    meta_html,
    path_html,
    image_output,
    score_buttons,
    notes_box,
    widgets.HTML('<b>Prompt</b>'),
    prompt_html,
    controls,
    message_html,
])

render_item()
display(panel)

In [ ]:
if QUALITY_SUMMARY_PATH.exists():
    print('Quality summary preview')
    display(pd.read_csv(QUALITY_SUMMARY_PATH))

if QUALITY_SCORES_PATH.exists():
    print('Quality scores preview')
    display(pd.read_csv(QUALITY_SCORES_PATH).head(20))

## Next Step

After saving your qualitative ratings, re-run the pipeline in evaluation-only mode so the main reports pick up the manual scores:

```bash
python run_pipeline.py --eval-only
```

That refreshes `evaluation_report.csv`, `summary.csv`, `evaluation_report.html`, and `summary.html` with the new quality columns.